Part 1: What is an LLM Gateway?
Think of an LLM Gateway as a smart middleware layer that sits between your application and multiple LLM providers (OpenAI, Anthropic, Google, Groq, Cohere, local models, etc.).

                    ┌─────────────────────────────┐
                    │       Your Application      │
                    │  (Chatbot, RAG, Agent, etc) │
                    └──────────────┬──────────────┘
                                   │
                                   ▼
                    ┌─────────────────────────────┐
                    │       LLM GATEWAY           │
                    │  • Routing                  │
                    │  • Fallbacks                │
                    │  • Caching                  │
                    │  • Rate Limiting            │
                    │  • Cost Tracking            │
                    │  • Observability            │
                    └──────┬─────┬─────┬─────┬────┘
                           │     │     │     │
                           ▼     ▼     ▼     ▼
                        OpenAI Claude Gemini Groq
                        
Without a Gateway (The Pain 😩)
Different SDKs and APIs for every provider

No fallback if one provider goes down

No central place to track costs

Hard to switch models without rewriting code

No caching → paying twice for the same query


⚙️ Part 2: Installation & Setup

We'll use:

LiteLLM → the most popular open-source LLM gateway (supports 100+ providers)

LangChain → for building agentic workflows on top of the gateway

python-dotenv → for managing API keys


In [21]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

from litellm import completion

In [22]:
import litellm
litellm.suppress_debug_info = True

In [23]:
import warnings
import logging

# Keep the recording clean — suppress noisy AWS-related warnings
warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

In [24]:
# Load API keys from a .env file
# Create a .env file in the same folder with:
# OPENAI_API_KEY=sk-...
# ANTHROPIC_API_KEY=sk-ant-...
# GROQ_API_KEY=gsk_...

import os
from dotenv import load_dotenv
load_dotenv()

# Quick check
print("OpenAI key loaded:    ", "✅" if os.getenv("OPENAI_API_KEY") else "❌")
print("gemini key loaded: ", "✅" if os.getenv("GOOGLE_API_KEY") else "❌")
print("Groq key loaded:      ", "✅" if os.getenv("GROQ_API_KEY") else "❌")

OpenAI key loaded:     ✅
gemini key loaded:  ✅
Groq key loaded:       ✅


🎯 Part 3: The Simplest LiteLLM Example — Unified API

The biggest pain point: every provider has a different SDK.

LiteLLM gives you one function — completion() — that works with all of them.

 Look at how clean this is

In [25]:
from litellm import completion

try:
    response = completion(
        model="gemini/gemini-2.0-flash",
        messages=[
            {"role": "user", "content": "Explain RAG in one sentence."}
        ]
    )
    print(response.choices[0].message.content)

except Exception as e:
    print(e)

litellm.RateLimitError: litellm.RateLimitError: geminiException - {
  "error": {
    "code": 429,
    "message": "You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 502.520248ms.",
    "status": "RESOURCE_EXHAUSTED",
    "details": [
      {
        "@type": "type.googleapis.com/google.rpc.Help",
        "links": [
          {
            "description": "Learn more about Gemin

In [30]:
from litellm import completion

# Same code, different providers — just change the `model` string!
# Call Groq (super fast inference)
response_groq = completion(
    model="groq/llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Explain RAG in one sentence."}]
)
print("🟢 Groq:      ", response_groq.choices[0].message.content)

BadRequestError: litellm.BadRequestError: GroqException - {"error":{"message":"Invalid API Key","type":"invalid_request_error","code":"invalid_api_key"}}
